The notebook extracts informations from a corpus to create a knowledge graph.
It is a Proof of concept of the pipeline and can be used as a demo.

In [1]:
from datetime import datetime
import hashlib
import json
from neo4j import GraphDatabase
import ollama
import os
from sentence_transformers import SentenceTransformer
import sys
import time
from tqdm import tqdm
sys.path.insert(1, "src/graph/")
from graph_builder import get_node_id, extract_graph, compute_chunk_embeddings, merge_graphs, validate_graph, build_neo4j_graph, feed_global_report, save_graph, load_to_neo4j, create_constraints
sys.path.insert(1, "src/preprocessing/")
from speeches import load_speeches, split_into_chunks, load_template_json, load_prompt_template, prompt_builder

KeyboardInterrupt: 

In [2]:
# Folder with speeches
#corpus = []
folder = os.path.join("data","discours-presidents")
example_json = load_template_json(
    os.path.join(
        "src", 
        "prompts",
        "prepare_llm_output_graph_production.json"
    )
)

# folder with the prompt used to extract informations
prompt_template = load_prompt_template(
    os.path.join(
        "src",
        "prompts",
        "prompt_RAG_production.txt"
    )
)
# Dict to save nodes and relationships
merged_graph = {
    "entities": [],
    "relations": []
}

In [3]:
with open(
    os.path.join(
        "src", "preprocessing","global_report.json"
        )
    ) as json_file:
    global_report = json.load(json_file)

In [4]:
with open("credential.json") as json_file:
    credential = json.load(json_file)

driver = GraphDatabase.driver(
    credential["URI"],
    auth=(
        credential["USER"], 
        credential["PASSWORD"]
    )
)

# Check connectivity with neo4j 
# (need to lunch Neo4j desktop and DB before)
driver.verify_connectivity()

create_constraints(
    driver,
    os.path.join("src", "graph", "graph_constraint.txt")
)

print("Neo4j is connected !")

Neo4j constraints created.
Neo4j is connected !


# Extraction pipeline

In [5]:
llm_model = "qwen2.5:7b"
log_dic_path = os.path.join("output", "log_graph.txt")
if os.path.exists(log_dic_path):
    with open(log_dic_path, "r", encoding="utf-8") as f:
        log_dic = json.load(f)
else:
    log_dic = {}

In [ ]:
# extract data from speech and sotres it into a list of dict with this fields : 
# 'id', 'date', 'title', 'filename', 'header', 'text', 'chunks'
corpus = load_speeches(folder)
for speech in tqdm(corpus[:50]):#remove comment to lunch on all dataset
#for speech in corpus[:1]:
#check if the file was alreday present into the database
    filename = speech["filename"]
    if filename in log_dic.keys() and log_dic[filename]["status"] == "SUCCESS" and log_dic[filename]["llm_model"] == llm_model:
        #file already analyzed
        continue
    start = time.time()
    try:
        # split text in chuncks and store it
        speech["chunks"] = split_into_chunks(speech)
        # embedding
        speech["chunks"] = compute_chunk_embeddings(speech["chunks"])
        # label and relations for the speech
        merged_graph = {
            "entities": [],
            "relations": []
        }
        # split speech and made embedding
        for chunk in speech["chunks"]:
                prompt = prompt_builder(
                    prompt_template, 
                    example_json, 
                    chunk["text"]
                )
                try:
                    graph = extract_graph(
                        prompt, 
                        model = llm_model
                    )
                except Exception as e:
                    print(
                        f"Chunk {chunk['chunk_id']} failed : {e}"
                    )
                    continue
                if graph is None:
                    continue

                merged_graph = merge_graphs(
                    merged_graph,
                    graph
                )
        # Validation
        # check duplicate and no valide elements
        merged_graph, report = validate_graph(merged_graph)
        global_report = feed_global_report(
            report, 
            speech, 
            global_report
        )
        # build neo4j graph
        neo4j_graph = build_neo4j_graph(
            speech,
            merged_graph
        )
        save_graph(
            neo4j_graph,
            os.path.join(
                "output", 
                "merged_graph",
                speech['id']+".json"
                )
        )
        save_graph(
            neo4j_graph,
            os.path.join(
                "output", 
                "neo4j_graph",
                speech['id']+".json"
                )
        )

        # LOAD Neo4j
        load_to_neo4j(
            driver,
            neo4j_graph
        )
    except Exception as e:
        elapsed = time.time() - start
        # update log SUCCESS
        log_dic[filename] = {
            "status": "SUCCESS",
            "llm_model": llm_model,
            "prompt_version": "v1",
            "processed_at": datetime.now().isoformat(),
            "processing_time": round(elapsed, 2),
            "n_chunks": len(speech["chunks"]),
            "n_entities": len(
                merged_graph["entities"]
            ),
            "n_relations": len(
                merged_graph["relations"]
            )
        }
    finally:
        with open(log_dic_path, "w", encoding="utf-8") as f:
            json.dump(log_dic, f, indent=2, ensure_ascii=False)

    # Save the final report
    with open("global_report.json", "w", encoding="utf-8") as f:
        json.dump(global_report, f, indent=2, ensure_ascii=False)

    driver.close()
    print("\nPipeline done.")

KeyError: 'evidence'

In [ ]:
with open("global_report.json", "w", encoding="utf-8") as f:
    json.dump(global_report, f, indent=2, ensure_ascii=False)

{'speeches': 2,
 'chunks': 40,
 'entities': 114,
 'relations': 92,
 'duplicate_entities': 0,
 'duplicate_relations': 0,
 'missing_targets': {'Dialogue': 1,
  'asséchement des richesses culturelles du monde': 1,
  'atteinte au droit élémentaire des enfants à vivre protégés des turpitudes de certains adultes': 1,
  'atteinte à notre sécurité et donc à notre liberté et à notre intégrité': 1,
  'richesses culturelles du monde': 1,
  'Article 3': 1,
  'Association': 1,
  'Difficulté': 1,
  'Décret': 1,
  'Formation aux auxiliaires de vie scolaire': 1,
  'Investissement financier': 1,
  'Loi de 2005': 1,
  'Person': 1,
  'mise en œuvre de la loi du 11 février 2005': 1,
  'rapport': 1},
 'missing_sources': {'Gouvernement': 1, 'Nous': 1, 'rapport': 1}}